# Exercise 1 - Google Earth Engine and Cloud Computing

Google Earth Engine is an online tool which provides access to vast amounts of geospatial data, as well as infrastructure for working with that data. This means that Google hosts all of the large data sets without you having to download them. It also means that the expensive processing happens on their computers, and that you can do complex analysis on a small computer and without lots of internet bandwidth.

For a list of available data sets on Google Earth Engine, you can visit this link: [https://developers.google.com/earth-engine/datasets](https://developers.google.com/earth-engine/datasets)

For a further introduction to Google Earth Engine, please see this link: [https://earthengine.google.com/](https://earthengine.google.com/)

### Goals
In this exercise, we will use Google Earth Engine to visualize different kinds of satellite data . This is useful if you want to know how things look from different satellite products, and across different kinds of spatial and climate data.

To do this, we will use the Python computer language, which we can link to Google Earth Engine. There are detailed instructions on how to set this up [here](https://developers.google.com/earth-engine/python_install).

As an alternative, you can run all of the same analysis we will do in [Google Colab](https://colab.research.google.com/). To do so, download the exercise file from [Github](https://github.com/tasmi/Workshop_Kathmandu_Feb2026) to your Google Colab instance. You can then follow the exercise in the same way as if you ran it on your own computer.

## Importing Python Modules

The first step to running any Python program is to import _modules_, which are small sets of commands that do specific tasks. In this exercise, we want to use three modules:

1. [Google Earth Engine](https://github.com/google/earthengine-api)
2. [Matplotlib](https://matplotlib.org/)
3. [Numpy](https://numpy.org/)

These will provide us access to Google's servers, utilities for plotting data, and utilities for doing mathematical operations.

To add modules to your Python program, use the _import_ line:

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import datetime

In [3]:
import ee
import geemap
project_name = 'workshop-488206'
#The first time you use the earthengine module, you need to link your account credentials. Afterwards, your computer stores the authentication file
ee.Authenticate()
#ee.Initialize()
ee.Initialize(project=project_name)
print(ee.__version__)

1.5.24


Now that we have access to those Python modules, we can test if they are working and we can access Google's servers:

In [4]:
dem = ee.Image('USGS/SRTMGL1_003') #Load in the global SRTM elevation data set
xy = ee.Geometry.Point([86.9250, 27.9881]) #Define the location of interest
elev = dem.sample(xy, 30).first().get('elevation').getInfo() #Sample the data set at that point
print('Mount Everest elevation (m):', elev) #Print the result

Mount Everest elevation (m): 8729


Those four lines of code do a few steps:

1. Import the global SRTM Elevation data set at 30 m (see details [here](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003))
2. Choose a point location based on latitude and longitude, which is defined as a 'Point Geometry'
3. Sample the elevation data at 30 m resolution, and use _.getInfo()_ to download the result
4. Print the result

The important thing here is that no processing is done on your computer, and data is only downloaded when you use _.getInfo()_. This means you can do complex operations without having to download anything but the final answer. We don't need to download the elevation data itself, we only want the elevation at a single point! Google Earth Engine takes care of the rest.

## Defining a Location of Interest

The main goal of this exercise is to choose a location of interest (for example, a glacier, a hydropower project, a weather station location, etc) and visualize different kinds of satellite data at that location. The main benefit here is that we can skip the step of figuring out how to download, process, and sample very large data sets, and only concentrate on visualizing the data we actually are interested in. In the second exercise today we will look at downloading data, but for now we will focus only on visualizations.

The easiest way to define a point location is with _latitude and longitude_, which you may already have from GPS measurements or known data about a climate station. Two easy alternative ways to get GPS coordinates is by using [Google Maps](https://maps.google.com/) or the website [geojson.io](https://geojson.io/).

<img src="https://raw.githubusercontent.com/tasmi/Workshop_Kathmandu_Feb2026/main/Notebooks/Images/Kathmandu.png" alt="" style="width: 600px;"/>

I can then use the chosen location (for example, Kathmandu) in the same code as above to check the elevation:

In [5]:
#NOTE THAT THIS IS [X, Y], and is opposite of the output from Google Maps!
ktm = ee.Geometry.Point([85.31823525656749, 27.713513421380526])
elev = dem.sample(ktm, 30).first().get('elevation').getInfo() #Sample the data set at that point
print('Kathmandu elevation (m):', elev) #Print the result

Kathmandu elevation (m): 1311


## Choosing Data

There is a **massive** number of data sets we can choose from on Google Earth Engine. If we look at the [data catalog](https://developers.google.com/earth-engine/datasets), we can search for 'Climate' data as a first step:

<img src="https://raw.githubusercontent.com/tasmi/Workshop_Kathmandu_Feb2026/main/Notebooks/Images/ClimateData.png" alt="" style="width: 600px;"/>

Let's choose a simple rainfall data set: [CHIRPS Daily Precipitation](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY)

This data covers the whole globe for the past 30+ years, and gives us daily total precipitation. We can add it to our Python script with the command:

In [6]:
rainfall = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") #Load in the precipitation data
data_length = rainfall.size().getInfo() #Check the data size (number of images)
print(data_length) #Print out the size of the data

16467


This means we have more than 16000 days of precipitation data that we can access through that one line of Python code! That is a lot of data.

We can select a smaller subset, for example the last year by using a _filter_.

In [7]:
rainfall = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate('2023-01-01', '2024-01-01') #Filter to only one year
data_length = rainfall.size().getInfo()
print(data_length)

365


Let's now look at visualizing that data. We will rely on a new module: __geemap__ ([https://geemap.org/](https://geemap.org/))

Using the **geemap** module, we can directly visualize the data we have selected.

NOTE: You may need to enable the ipyleaflet extension if the below code does not run:

    jupyter nbextension enable --py --sys-prefix ipyleaflet

Let's start with just one image first:

In [8]:
single_day = rainfall.first() #Select one day of rainfall data
#single_day = single_day.updateMask(single_day.gt(0.1)) #Optionally mask out low rainfall

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=6) #Set up a map
image_viz_params = {'palette': ['00204C', 'FFE945'],'min': 0.1,'max': 5, 'opacity':0.9} #Add visualization parameters
Map.addLayer(single_day, image_viz_params) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map

Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

We can also modify the color palette using many different parameters. See here: [https://developers.google.com/earth-engine/guides/image_visualization](https://developers.google.com/earth-engine/guides/image_visualization) for more details.

The main tricky thing with colors is that Earth Engine uses *hex codes*, which are a special color format. Fortunately, we can convert any color to a hex code with various online tools, such as this one: [https://htmlcolorcodes.com/](https://htmlcolorcodes.com/). Let's try modifying the color palette and plotting again.

In [9]:
single_day = rainfall.first() #Select one day of rainfall data
#single_day = single_day.updateMask(single_day.gt(0.1)) #Optionally mask out low rainfall

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=6) #Set up a map
image_viz_params = {'palette': ['1C1C1C', '4B4EBD', 'E80E0E'],'min': 0.1,'max': 5, 'opacity':0.9} #Add visualization parameters
Map.addLayer(single_day, image_viz_params) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map

Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

Note that I added a third color to the color ramp -- you can make a simple or more complicated color ramp in this way.

Let's go one step further and visualize a long-term average precipitation instead of one day. To do this, we can replace our 'single_day' with a new line of code:

In [10]:
single_day = rainfall.mean() #Take the annual mean rainfall instead of a single day
#single_day = rainfall.sum() #Take the annual sum rainfall instead of a single day
#single_day = single_day.updateMask(single_day.gt(0.1)) #Optionally mask out low rainfall

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=6) #Set up a map
image_viz_params = {'palette': ['1C1C1C', '4B4EBD', 'E80E0E'],'min': 0.1,'max': 5, 'opacity':0.9} #Add visualization parameters
Map.addLayer(single_day, image_viz_params) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map

Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

This takes a little bit more time to run, but not very much! We just took the average of a whole year of rainfall data and plotted it by changing one line. None of this computation happened on our laptops, but rather on Google's servers. In this way we can access and explore very large data sets quite quickly!

Let's try another example, this time using a **multi-band image**. Many satellites, such as Landsat and Sentinel-2, have several different spectral bands (red, green, blue, etc). By visualizing different bands, we can understand more details about the landscape.

We can start by picking out the Landsat 9 data collection (the most recent Landsat satellite) and filtering it just like we did with rainfall:

In [11]:
landsat = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(ktm) #Note that I filter to only find images that overlap with Kathmandu
data_length = landsat.size().getInfo()
print(data_length)

90


In [12]:
single_image = landsat.first()

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=10) #Set up a map, this time more zoomed in
Map.addLayer(single_image) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map

Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

This gives us a similar map, but we don't know which Landsat band we are looking at! If we want to check which bands are available, we can use *.getInfo()* to check:

In [13]:
single_image.getInfo()

{'type': 'Image',
 'bands': [{'id': 'SR_B1',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [7631, 7831],
   'crs': 'EPSG:32645',
   'crs_transform': [30, 0, 258285, 0, -30, 3151815]},
  {'id': 'SR_B2',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [7631, 7831],
   'crs': 'EPSG:32645',
   'crs_transform': [30, 0, 258285, 0, -30, 3151815]},
  {'id': 'SR_B3',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [7631, 7831],
   'crs': 'EPSG:32645',
   'crs_transform': [30, 0, 258285, 0, -30, 3151815]},
  {'id': 'SR_B4',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [7631, 7831],
   'crs': 'EPSG:32645',
   'crs_transform': [30, 0, 258285, 0, -30, 3151815]},
  {'id': 'SR_B5',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
 

Now we can see that we have many different bands. These correspond to the bands described here: [https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2)

We can set a 'true color' composite by selecting the red, green, and blue bands to visualize:

In [14]:
#Correct for the data scaling (see here for details: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2)
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

single_image = landsat.first() #Take the first image in our collection
single_image = apply_scale_factors(single_image) #Apply the scale factors
visualization = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min':0, 'max':0.3} #Choose the true-color bands

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=10)
Map.addLayer(single_image, visualization) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map


Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

We can also look at other color composites -- these are often used to understand different features in the landscape. For example, if we use bands 5,4,3, that gets us a visualization that emphasizes vegetation:

In [15]:
single_image = landsat.first() #Take the first image in our collection
single_image = apply_scale_factors(single_image) #Apply the scale factors
visualization = {'bands': ['SR_B5', 'SR_B4', 'SR_B3'], 'min':0, 'max':0.3} #Choose false-color bands

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=10)
Map.addLayer(single_image, visualization) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map

Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

For more information on different Landsat band combinations, there are many online resources. For example: [https://www.esri.com/arcgis-blog/products/product/imagery/band-combinations-for-landsat-8](https://www.esri.com/arcgis-blog/products/product/imagery/band-combinations-for-landsat-8)

Let's try one more that emphasizes urban areas:

In [16]:
single_image = landsat.first() #Take the first image in our collection
single_image = apply_scale_factors(single_image) #Apply the scale factors
visualization = {'bands': ['SR_B7', 'SR_B6', 'SR_B4'], 'min':0, 'max':0.3} #Choose false-color bands

Map = geemap.Map(center=[27.713513421380526,85.31823525656749], zoom=10)
Map.addLayer(single_image, visualization) #Plot the map
Map.addLayer(ktm) #Plot the center point

Map

Map(center=[27.713513421380526, 85.31823525656749], controls=(WidgetControl(options=['position', 'transparent_…

Now we know how to visualize single images as either one color ramp or using multiple bands. We will use these types of visualizations throughout the day to explore the different analyses we complete without having to download lots of data.